In [11]:
import numpy as np
import math
import matplotlib.pyplot as plt
import os
import pandas as pd

# Load the data we just exported
df = pd.read_csv("../data/blade_design_input.csv")

# ==========================================
# 1. INPUT PARAMETERS
# ========================================== 

# Extract the columns into the lists the loop expects
stator_or_rotor = df['type'].tolist()
radial_section  = df['section'].tolist()
chord_length    = df['chord'].tolist()
chi_in          = df['chi_in'].tolist()
chi_out         = df['chi_out'].tolist()
gamma           = df['gamma'].tolist()
t_max_target = 0.15

naca_data = np.array([
    [0.0000, 0.00000], [0.0050, 0.00973], [0.0075, 0.01173], [0.0125, 0.01492],
    [0.0250, 0.02078], [0.0500, 0.02895], [0.0750, 0.03504], [0.1000, 0.03994],
    [0.1500, 0.04747], [0.2000, 0.05287], [0.2500, 0.05664], [0.3000, 0.05901],
    [0.3500, 0.05995], [0.4000, 0.05957], [0.4500, 0.05792], [0.5000, 0.05517],
    [0.5500, 0.05148], [0.6000, 0.04700], [0.6500, 0.04186], [0.7000, 0.03621],
    [0.7500, 0.03026], [0.8000, 0.02426], [0.8500, 0.01826], [0.9000, 0.01225],
    [0.9500, 0.00625], [1.0000, 0.00025]
])

# Create directories if they don't exist
os.makedirs("../images", exist_ok=True)
os.makedirs("../blade_geometry", exist_ok=True)

# ==========================================
# 2. GEOMETRY GENERATION FUNCTIONS
# ==========================================

def rotate_coords(x, y, angle_deg):
    rad = math.radians(angle_deg)
    xr = x * math.cos(rad) - y * math.sin(rad)
    yr = x * math.sin(rad) + y * math.cos(rad)
    return xr, yr

def generate_blade_geometry(chord, naca_coords, c_in, c_out, g, thickness_ratio):
    x_naca = naca_coords[:, 0]
    yt = naca_coords[:, 1] * (thickness_ratio / 0.12)
    
    chi_in_rel = math.radians(c_in - g)
    B = math.tan(chi_in_rel)
    A = -B 
    
    xu, yu, xl, yl, xc_line, yc_line = [], [], [], [], [], []
    
    for i in range(len(x_naca)):
        x = x_naca[i]
        yc = (A * x**2 + B * x) * chord
        xc = x * chord
        theta = math.atan(2 * A * x + B)
        
        xu.append(xc - (yt[i] * chord * math.sin(theta)))
        yu.append(yc + (yt[i] * chord * math.cos(theta)))
        xl.append(xc + (yt[i] * chord * math.sin(theta)))
        yl.append(yc - (yt[i] * chord * math.cos(theta)))
        xc_line.append(xc)
        yc_line.append(yc)
        
    xu_r, yu_r = rotate_coords(np.array(xu), np.array(yu), g)
    xl_r, yl_r = rotate_coords(np.array(xl), np.array(yl), g)
    xc_r, yc_r = rotate_coords(np.array(xc_line), np.array(yc_line), g)
        
    return xu_r, yu_r, xl_r, yl_r, xc_r, yc_r

def calculate_area_integration(xu, yu, xl, yl):
    """
    Calculates the area using the Trapezoidal Rule.
    Note: Coordinates should be in the un-rotated frame for 
    the simplest 'y_upper - y_lower' calculation.
    """
    # Area under the upper surface
    area_upper = np.trapz(yu, xu)
    
    # Area under the lower surface
    area_lower = np.trapz(yl, xl)
    
    # The area of the blade is the difference between them
    # We use absolute value because depending on coordinate 
    # direction (LE to TE), one integral might be negative.
    return abs(area_upper - area_lower)

# ==========================================
# 3. MAIN LOOP THROUGH ALL SECTIONS
# ==========================================

for i in range(len(stator_or_rotor)):
    # Extract current parameters
    comp_type = stator_or_rotor[i]
    loc = radial_section[i]
    c = chord_length[i]
    cin = chi_in[i]
    cout = chi_out[i]
    g = gamma[i]
    
    # Generate Geometry
    xu, yu, xl, yl, xc, yc = generate_blade_geometry(c, naca_data, cin, cout, g, t_max_target)
    
    # calculate area of the blade profile
    blade_area = calculate_area_integration(xu, yu, xl, yl)

    # --- PLOTTING ---
    plt.figure(figsize=(10, 6))

    plt.plot(xu, yu, 'r-', linewidth=2, label='Suction Side')
    plt.plot(xl, yl, 'b-', linewidth=2, label='Pressure Side')
    plt.plot(xc, yc, 'k--', linewidth=1.5, label='Camber Line')

    # Chord Line
    c_line_x, c_line_y = rotate_coords(np.array([0, c]), np.array([0, 0]), g)
    plt.plot(c_line_x, c_line_y, 'g:', linewidth=2,
            label=f'Chord Line (Stagger={g:.1f}°)')

    # Metal Angle Indicators
    vec_len = c * 0.15
    plt.arrow(xc[0], yc[0],
            math.cos(math.radians(cin))*vec_len,
            math.sin(math.radians(cin))*vec_len,
            color='blue', width=0.0002, head_width=0.001)

    plt.arrow(xc[-1], yc[-1],
            math.cos(math.radians(cout))*vec_len,
            math.sin(math.radians(cout))*vec_len,
            color='purple', width=0.0002, head_width=0.001)

    plt.axis('equal')
    plt.grid(True, linestyle=':', alpha=0.5)

    plt.title(
        f"{comp_type} - {loc} Section\n"
        f"Chord: {c*1000:.1f} mm | Axial Chord: {c*math.cos(math.radians(g))*1000:.2f} mm | Area: {blade_area*1e6:.2f} mm$^2$",
        fontsize=22
    )

    plt.xlabel("Axial Direction (m)", fontsize=20)
    plt.ylabel("Tangential Direction (m)", fontsize=20)

    plt.xticks(fontsize=16)
    plt.yticks(fontsize=16)
    plt.tight_layout()

    plt.legend(loc='best', fontsize=16)

    plt.gca().set_aspect('equal', adjustable='box')

    # Save Plot
    plt.savefig(f"../images/v4/{comp_type}_{loc}.png", dpi=300, bbox_inches='tight')
    plt.close()

    
    # --- EXPORT TO CSV ---
    x_coords = np.concatenate([xu, xl[::-1]])
    y_coords = np.concatenate([yu, yl[::-1]])
    geometry_data = np.column_stack((x_coords, y_coords))

    file_name = f"../blade_geometry/v4/{comp_type}_{loc}.csv"
    np.savetxt(file_name, geometry_data, delimiter=",", header="X_Coordinate,Y_Coordinate", comments='')

    print(f"Processed: {comp_type} {loc}")

print("\nAll 6 sections successfully generated.")

C:\Users\User\AppData\Local\Temp\ipykernel_28208\4274674764.py:83: DeprecationWarning: `trapz` is deprecated. Use `trapezoid` instead, or one of the numerical integration functions in `scipy.integrate`.
  area_upper = np.trapz(yu, xu)
C:\Users\User\AppData\Local\Temp\ipykernel_28208\4274674764.py:86: DeprecationWarning: `trapz` is deprecated. Use `trapezoid` instead, or one of the numerical integration functions in `scipy.integrate`.
  area_lower = np.trapz(yl, xl)


Processed: Stator Hub
Processed: Stator Mean
Processed: Stator Tip
Processed: Rotor Hub
Processed: Rotor Mean
Processed: Rotor Tip

All 6 sections successfully generated.
